## **Problem Statement**

Dream Housing Finance wants to automate real-time loan approval
decisions. They've collected applicant data (income, credit history,
education, etc.) and want a model that predicts Loan_Status (Y/N) for new
applicants in the test set.

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
df = pd.read_csv("./data/train.csv")

**Data Inspect**

In [33]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [30]:
df.shape

(614, 13)

In [31]:
df.dtypes

Loan_ID                  str
Gender                   str
Married                  str
Dependents               str
Education                str
Self_Employed            str
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area            str
Loan_Status              str
dtype: object

In [9]:
df.columns.tolist()

['Loan_ID',
 'Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Loan_Amount_Term',
 'Credit_History',
 'Property_Area',
 'Loan_Status']

**Data Audit**

In [32]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

Credit History is serverly affected by null values (50 rows of null). It's column where 1 represents follows credit guidelines and 0 means otherwise - It's one of the most important columns in the whole dataset. We don't 32 values for the self employed column - I think it's also important because without employment status at all it's hard to give our loans and loan amount, loan amount term also has a bunch of null value without them we definitely can't decide whether to approve or not. others like dependents also play a vital role as more dependents means you need more income to run the family and hence there is hight chance of loan default unless the income stabilizes it. Same goes for married. gender has some context to it. Mostly in the current generation men and women are treated as equals and they both go to jobs. I think in a more general context it doesn't make much of a difference. 

In [22]:
for col in df.columns:
    if str(col) != "Loan_ID":
        print(f"{col} : {df[col].unique()}\n")

Gender : <StringArray>
['Male', 'Female', nan]
Length: 3, dtype: str

Married : <StringArray>
['No', 'Yes', nan]
Length: 3, dtype: str

Dependents : <StringArray>
['0', '1', '2', '3+', nan]
Length: 5, dtype: str

Education : <StringArray>
['Graduate', 'Not Graduate']
Length: 2, dtype: str

Self_Employed : <StringArray>
['No', 'Yes', nan]
Length: 3, dtype: str

ApplicantIncome : [ 5849  4583  3000  2583  6000  5417  2333  3036  4006 12841  3200  2500
  3073  1853  1299  4950  3596  3510  4887  2600  7660  5955  3365  3717
  9560  2799  4226  1442  3750  4166  3167  4692  3500 12500  2275  1828
  3667  3748  3600  1800  2400  3941  4695  3410  5649  5821  2645  4000
  1928  3086  4230  4616 11500  2708  2132  3366  8080  3357  3029  2609
  4945  5726 10750  7100  4300  3208  1875  4755  5266  1000  3333  3846
  2395  1378  3988  2366  8566  5695  2958  6250  3273  4133  3620  6782
  2484  1977  4188  1759  4288  4843 13650  4652  3816  3052 11417  7333
  3800  2071  5316  2929  3572  745

Apart from null values, these are the problems we have an Encoding problem. The 'dependents' columns has a value of 3+ and if we go ahead and treat it just like 3. We would be modeling wrong. 

In [21]:
df['Loan_Status'].value_counts()

Loan_Status
Y    422
N    192
Name: count, dtype: int64

In [23]:
print(422/(422+192))
print(192/(422+192))

0.6872964169381107
0.3127035830618892


There is some class imbalance. Class-Y has 68.7% and Class-N has 31.3% of the data points.

In [27]:
df[df["Loan_ID"].duplicated()].shape

(0, 13)

There are no duplicate rows.